In [1]:
from pyspark.sql import SparkSession
import getpass
username =getpass.getuser()
spark =SparkSession.builder.config('spark.ui.port','0') \
.config('spark.shuffle.useOldFetchProtocol','true') \
.config('spark.sql.warehouse.dir', f'/user/{username}/warehouse') \
.enableHiveSupport() \
.master('yarn') \
.getOrCreate()

In [2]:
from pyspark.sql.functions import current_timestamp,when,col

In [3]:
loans_repay_schema='loan_id string,total_principal_recieved float,total_interest_recieved float,total_late_fee float,total_payment float,last_payment_amount float,last_payment_date string, next_payment_date string'

In [4]:
loan_repayments_df = spark.read \
.format("csv") \
.schema(loans_repay_schema) \
.option("header",True) \
.load(f"/user/{username}/lendingclub/raw/loans_repayment_data_csv")

In [5]:
loan_repayments_df

loan_id,total_principal_recieved,total_interest_recieved,total_late_fee,total_payment,last_payment_amount,last_payment_date,next_payment_date
491699,7000.0,1444.94,0.0,8444.942,954.65,Dec-2012,null
491685,15000.0,2583.58,0.0,17583.584,517.23,Mar-2013,null
491667,6400.0,807.51,0.0,7207.5093,209.97,Mar-2013,null
491160,4000.0,963.31,0.0,4963.3066,144.87,Mar-2013,null
491675,20000.0,2887.0,0.0,22886.992,16.27,Jul-2011,null
491668,1773.57,591.33,0.0,2957.37,394.94,Mar-2011,null
491663,5500.0,235.4,0.0,5735.4,2147.02,Sep-2010,null
491632,10000.0,1466.31,0.0,11466.3125,7274.73,Apr-2011,null
491618,24999.99,2085.75,0.0,27085.738,21866.75,Oct-2010,null
491622,25000.0,13079.81,0.0,38079.81,7912.97,Feb-2015,null


In [6]:
(loan_repayments_df.filter(loan_repayments_df.total_principal_recieved.isNull())).count()

69

In [7]:
columns_to_check = ['total_principal_recieved','total_interest_recieved','total_late_fee','total_payment','last_payment_amount']

In [8]:
loan_repayments_filtered = loan_repayments_df.na.drop(subset=columns_to_check)

In [9]:
loan_repayments_filtered.filter(loan_repayments_filtered.total_payment == 0.0)

loan_id,total_principal_recieved,total_interest_recieved,total_late_fee,total_payment,last_payment_amount,last_payment_date,next_payment_date
485818,14640.096,13388.84,13000.0,0.0,0.0,0.0,Mar-2013
485471,29620.818,29134.64,25000.0,0.0,0.0,0.0,Mar-2013
482256,8735.611,7479.87,8000.0,0.0,0.0,0.0,Feb-2011
478160,410.0,407.36,0.0,0.0,143.1,410.0,null
476557,28865.18,24164.67,5692.31,0.0,6972.59,19916.78,Dec-2010
472516,25951.482,24731.76,25000.0,0.0,0.0,0.0,May-2010
472197,12048.13,12018.01,10000.0,0.0,0.0,0.0,Jan-2013
467364,29216.791,29066.19,24250.0,0.0,0.0,0.0,Dec-2012
399499,26557.729,26336.41,24000.0,0.0,0.0,0.0,Dec-2010
451482,7587.5513,7587.55,7000.0,0.0,0.0,0.0,Jan-2011


In [10]:
loan_repayments_filtered.filter(loan_repayments_filtered.total_payment == 0.0).count()

995

In [11]:
loan_repayments_fixed = loan_repayments_filtered.withColumn(
    "total_payment",
    when(
        (col("total_principal_recieved") != 0.0) &
        (col("total_payment") == 0.0),
        col("total_principal_recieved") + col("total_interest_recieved") + col("total_late_fee")
    ).otherwise(col("total_payment"))
)

In [12]:
loan_repayments_fixed

loan_id,total_principal_recieved,total_interest_recieved,total_late_fee,total_payment,last_payment_amount,last_payment_date,next_payment_date
491699,7000.0,1444.94,0.0,8444.942,954.65,Dec-2012,null
491685,15000.0,2583.58,0.0,17583.584,517.23,Mar-2013,null
491667,6400.0,807.51,0.0,7207.5093,209.97,Mar-2013,null
491160,4000.0,963.31,0.0,4963.3066,144.87,Mar-2013,null
491675,20000.0,2887.0,0.0,22886.992,16.27,Jul-2011,null
491668,1773.57,591.33,0.0,2957.37,394.94,Mar-2011,null
491663,5500.0,235.4,0.0,5735.4,2147.02,Sep-2010,null
491632,10000.0,1466.31,0.0,11466.3125,7274.73,Apr-2011,null
491618,24999.99,2085.75,0.0,27085.738,21866.75,Oct-2010,null
491622,25000.0,13079.81,0.0,38079.81,7912.97,Feb-2015,null


In [13]:
# Remove rows with no 0s
loans_payments_fixed = loan_repayments_fixed.filter("total_payment != 0.0")

In [14]:
loans_payments_fixed.filter("last_payment_date = 0.0").count()

48

In [15]:
loans_payments_fixed.filter("next_payment_date = 0.0").count()

24

In [16]:
#Fixing where dates are 0
loans_payments_fixed_ldate = loans_payments_fixed.withColumn(
    "last_payment_date",
    when(
        (col("last_payment_date") == 0.0),
        None
    ).otherwise(col("last_payment_date"))
)

In [17]:
loans_payments_fixed_ndate = loans_payments_fixed_ldate.withColumn(
    "next_payment_date",
    when(
        (col("next_payment_date") == 0.0),
        None
    ).otherwise(col("last_payment_date"))
)

In [18]:
loans_payments_fixed_ndate.filter("last_payment_date = 0.0").count()

0

In [22]:
loans_payments_fixed_ndate.write \
.option("header",True) \
.format("csv") \
.mode("overwrite") \
.option("path",f"/user/{username}/lendingclub/cleaned/loans_repayment_csv") \
.save()

In [24]:
loans_payments_fixed_ndate.write \
.format("parquet") \
.option("path",f"/user/{username}/lendingclub/cleaned/loans_repayment_parquet") \
.save()